<a href="https://colab.research.google.com/github/Sasidhar015/Data-engineering-training-2026/blob/main/Advance_Pyspark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip -q install pyspark

In [2]:
from google.colab import files
uploaded = files.upload()

Saving machine_configurations.json to machine_configurations.json
Saving machine_master.csv to machine_master.csv
Saving manufacturing_production_120k.csv to manufacturing_production_120k (1).csv
Saving manufacturing_production_extra_12k.csv to manufacturing_production_extra_12k.csv
Saving plant_master.csv to plant_master.csv


In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("AdvancedPySparkPractice") \
    .getOrCreate()

df = spark.read.csv("/content/manufacturing_production_120k.csv",
    header=True,
    inferSchema=True)


In [4]:
# Display the first 20 rows
df.show(20, truncate=False)

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|shift  |product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|PR0000001    |2026-08-07     |PL03    |MC011     |OP0275     |Evening|MTR-100     |Motors        |1060         |993           |30             |34              |388.11    |61.4         |3.88          |Rework        |No              |Charan    |
|PR0000002    |2026-

In [5]:
# Display the DataFrame schema
df.printSchema()

root
 |-- production_id: string (nullable = true)
 |-- production_date: date (nullable = true)
 |-- plant_id: string (nullable = true)
 |-- machine_id: string (nullable = true)
 |-- operator_id: string (nullable = true)
 |-- shift: string (nullable = true)
 |-- product_code: string (nullable = true)
 |-- product_family: string (nullable = true)
 |-- units_planned: integer (nullable = true)
 |-- units_produced: integer (nullable = true)
 |-- defective_units: integer (nullable = true)
 |-- downtime_minutes: integer (nullable = true)
 |-- energy_kwh: double (nullable = true)
 |-- temperature_c: double (nullable = true)
 |-- vibration_mm_s: double (nullable = true)
 |-- quality_status: string (nullable = true)
 |-- maintenance_flag: string (nullable = true)
 |-- supervisor: string (nullable = true)



In [6]:
# Display all column names
print(df.columns)

['production_id', 'production_date', 'plant_id', 'machine_id', 'operator_id', 'shift', 'product_code', 'product_family', 'units_planned', 'units_produced', 'defective_units', 'downtime_minutes', 'energy_kwh', 'temperature_c', 'vibration_mm_s', 'quality_status', 'maintenance_flag', 'supervisor']


In [7]:
# Count total records
total_records = df.count()
print("Total number of records:", total_records)

Total number of records: 120000


In [8]:
selected_df = df.select(
    "production_id",
    "plant_id",
    "machine_id",
    "product_family",
    "units_produced")
selected_df.show(10, truncate=False)

+-------------+--------+----------+--------------+--------------+
|production_id|plant_id|machine_id|product_family|units_produced|
+-------------+--------+----------+--------------+--------------+
|PR0000001    |PL03    |MC011     |Motors        |993           |
|PR0000002    |PL04    |MC019     |Controllers   |836           |
|PR0000003    |PL01    |MC002     |Motors        |380           |
|PR0000004    |PL03    |MC013     |Pumps         |720           |
|PR0000005    |PL04    |MC020     |Pumps         |296           |
|PR0000006    |PL01    |MC002     |Motors        |795           |
|PR0000007    |PL02    |MC007     |Valves        |955           |
|PR0000008    |PL02    |MC009     |Valves        |895           |
|PR0000009    |PL03    |MC015     |Controllers   |289           |
|PR0000010    |PL02    |MC006     |Controllers   |527           |
+-------------+--------+----------+--------------+--------------+
only showing top 10 rows


In [9]:
# Display column names and data types
for column_name, data_type in df.dtypes:
    print(column_name, ":", data_type)

production_id : string
production_date : date
plant_id : string
machine_id : string
operator_id : string
shift : string
product_code : string
product_family : string
units_planned : int
units_produced : int
defective_units : int
downtime_minutes : int
energy_kwh : double
temperature_c : double
vibration_mm_s : double
quality_status : string
maintenance_flag : string
supervisor : string


In [10]:
# Select a sample and display 10 records
sample_df = df.sample(withReplacement=False,fraction=0.1,seed=42)
sample_df.show(10, truncate=False)

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|shift  |product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|PR0000008    |2026-08-17     |PL02    |MC009     |OP0106     |Night  |VLV-320     |Valves        |862          |895           |51             |77              |266.06    |57.1         |1.34          |Reject        |No              |Irfan     |
|PR0000017    |2026-

In [11]:
# Create a Python list of tuples
plant_data = [("PL01", "Plant Alpha", "Hyderabad"),
    ("PL02", "Plant Beta", "Chennai"),
    ("PL03", "Plant Gamma", "Bengaluru")]

plant_df = spark.createDataFrame(plant_data,["plant_id", "plant_name", "city"])

plant_df.show(truncate=False)

+--------+-----------+---------+
|plant_id|plant_name |city     |
+--------+-----------+---------+
|PL01    |Plant Alpha|Hyderabad|
|PL02    |Plant Beta |Chennai  |
|PL03    |Plant Gamma|Bengaluru|
+--------+-----------+---------+



In [13]:
# Create a DataFrame with explicit column names
column_names = ["plant_id", "plant_name", "city"]

plant_df_explicit = spark.createDataFrame(plant_data,schema=column_names)

plant_df_explicit.show(truncate=False)

+--------+-----------+---------+
|plant_id|plant_name |city     |
+--------+-----------+---------+
|PL01    |Plant Alpha|Hyderabad|
|PL02    |Plant Beta |Chennai  |
|PL03    |Plant Gamma|Bengaluru|
+--------+-----------+---------+



In [15]:
from pyspark.sql.types import (StructType,StructField,StringType)

plant_schema = StructType([
    StructField("plant_id", StringType(), True),
    StructField("plant_name", StringType(), True),
    StructField("city", StringType(), True)])
plant_data = [
    ("PL01", "Plant Alpha", "Hyderabad"),
    ("PL02", "Plant Beta", "Chennai"),
    ("PL03", "Plant Gamma", "Bengaluru")]
plant_df_schema = spark.createDataFrame(plant_data,schema=plant_schema)

plant_df_schema.show(truncate=False)

+--------+-----------+---------+
|plant_id|plant_name |city     |
+--------+-----------+---------+
|PL01    |Plant Alpha|Hyderabad|
|PL02    |Plant Beta |Chennai  |
|PL03    |Plant Gamma|Bengaluru|
+--------+-----------+---------+



In [17]:
# Print the DataFrame schema
plant_df_schema.printSchema()

root
 |-- plant_id: string (nullable = true)
 |-- plant_name: string (nullable = true)
 |-- city: string (nullable = true)



In [19]:
# Add a constant country column

from pyspark.sql.functions import lit
plant_df_final = plant_df_schema.withColumn("country",lit("India"))
plant_df_final.show(truncate=False)

+--------+-----------+---------+-------+
|plant_id|plant_name |city     |country|
+--------+-----------+---------+-------+
|PL01    |Plant Alpha|Hyderabad|India  |
|PL02    |Plant Beta |Chennai  |India  |
|PL03    |Plant Gamma|Bengaluru|India  |
+--------+-----------+---------+-------+



In [21]:
import pandas as pd
pdf = pd.read_csv("/content/manufacturing_production_120k.csv")


In [22]:
print(pdf.head(5))

  production_id production_date plant_id machine_id operator_id    shift  \
0     PR0000001      2026-08-07     PL03      MC011      OP0275  Evening   
1     PR0000002      2026-08-03     PL04      MC019      OP0489  Morning   
2     PR0000003      2026-05-20     PL01      MC002      OP0189  Evening   
3     PR0000004      2026-01-15     PL03      MC013      OP0076  Morning   
4     PR0000005      2026-09-21     PL04      MC020      OP0501  Morning   

  product_code product_family  units_planned  units_produced  defective_units  \
0      MTR-100         Motors           1060             993               30   
1      CTL-330    Controllers            809             836                0   
2      MTR-300         Motors            442             380               17   
3      PMP-210          Pumps            787             720                7   
4      PMP-110          Pumps            344             296                7   

   downtime_minutes  energy_kwh  temperature_c  vibratio

In [24]:
print(pdf.tail(5))

       production_id production_date plant_id machine_id operator_id    shift  \
119995     PR0119996      2026-07-13     PL05      MC022      OP0193  Evening   
119996     PR0119997      2026-04-26     PL05      MC024      OP0124  Evening   
119997     PR0119998      2026-02-01     PL06      MC027      OP0051  Evening   
119998     PR0119999      2026-01-17     PL01      MC001      OP0452  Evening   
119999     PR0120000      2026-08-16     PL02      MC007      OP0420  Morning   

       product_code product_family  units_planned  units_produced  \
119995      MTR-200         Motors            435             399   
119996      CTL-130    Controllers            813             777   
119997      VLV-120         Valves            650             597   
119998      MTR-300         Motors            887             899   
119999      MTR-200         Motors            547             477   

        defective_units  downtime_minutes  energy_kwh  temperature_c  \
119995                1   

In [25]:
print("Shape of the DataFrame:", pdf.shape)
print("Total rows:", pdf.shape[0])
print("Total columns:", pdf.shape[1])

Shape of the DataFrame: (120000, 18)
Total rows: 120000
Total columns: 18


In [26]:
print(pdf.columns)

Index(['production_id', 'production_date', 'plant_id', 'machine_id',
       'operator_id', 'shift', 'product_code', 'product_family',
       'units_planned', 'units_produced', 'defective_units',
       'downtime_minutes', 'energy_kwh', 'temperature_c', 'vibration_mm_s',
       'quality_status', 'maintenance_flag', 'supervisor'],
      dtype='object')


In [27]:
print(pdf.dtypes)

production_id        object
production_date      object
plant_id             object
machine_id           object
operator_id          object
shift                object
product_code         object
product_family       object
units_planned         int64
units_produced        int64
defective_units       int64
downtime_minutes    float64
energy_kwh          float64
temperature_c       float64
vibration_mm_s      float64
quality_status       object
maintenance_flag     object
supervisor           object
dtype: object


In [32]:
selected_pdf = pdf[["plant_id", "machine_id", "units_produced"]]
print(selected_pdf.head(10))

  plant_id machine_id  units_produced
0     PL03      MC011             993
1     PL04      MC019             836
2     PL01      MC002             380
3     PL03      MC013             720
4     PL04      MC020             296
5     PL01      MC002             795
6     PL02      MC007             955
7     PL02      MC009             895
8     PL03      MC015             289
9     PL02      MC006             527


In [34]:
renamed_pdf = pdf.rename(columns={"units_produced": "production_output"})
print(renamed_pdf.head())

  production_id production_date plant_id machine_id operator_id    shift  \
0     PR0000001      2026-08-07     PL03      MC011      OP0275  Evening   
1     PR0000002      2026-08-03     PL04      MC019      OP0489  Morning   
2     PR0000003      2026-05-20     PL01      MC002      OP0189  Evening   
3     PR0000004      2026-01-15     PL03      MC013      OP0076  Morning   
4     PR0000005      2026-09-21     PL04      MC020      OP0501  Morning   

  product_code product_family  units_planned  production_output  \
0      MTR-100         Motors           1060                993   
1      CTL-330    Controllers            809                836   
2      MTR-300         Motors            442                380   
3      PMP-210          Pumps            787                720   
4      PMP-110          Pumps            344                296   

   defective_units  downtime_minutes  energy_kwh  temperature_c  \
0               30              34.0      388.11          61.40   
1     

In [37]:
rejected_pdf = pdf[pdf["quality_status"] == "Reject"]
print(rejected_pdf.head(10))
print("Total rejected records:", len(rejected_pdf))

   production_id production_date plant_id machine_id operator_id    shift  \
1      PR0000002      2026-08-03     PL04      MC019      OP0489  Morning   
2      PR0000003      2026-05-20     PL01      MC002      OP0189  Evening   
7      PR0000008      2026-08-17     PL02      MC009      OP0106    Night   
9      PR0000010      2026-02-13     PL02      MC006      OP0593  Evening   
21     PR0000022      2026-03-03     PL02      MC009      OP0294  Evening   
22     PR0000023      2026-10-15     PL04      MC019      OP0187  Morning   
23     PR0000024      2026-06-07     PL03      MC013      OP0456  Morning   
28     PR0000029      2026-04-30     PL05      MC023      OP0339    Night   
30     PR0000031      2026-01-04     PL01      MC003      OP0524  Evening   
45     PR0000046      2026-04-02     PL06      MC027      OP0484  Morning   

   product_code product_family  units_planned  units_produced  \
1       CTL-330    Controllers            809             836   
2       MTR-300       

In [39]:
high_temp_pdf = pdf[pdf["temperature_c"] > 75]
print(high_temp_pdf.head(10))
print("Total high-temperature records:", len(high_temp_pdf))

    production_id production_date plant_id machine_id operator_id    shift  \
14      PR0000015      2026-03-15     PL02      MC007      OP0238  Morning   
16      PR0000017      2026-08-04     PL03      MC011      OP0533  Morning   
19      PR0000020      2026-02-18     PL03      MC014      OP0563  Morning   
26      PR0000027      2026-01-03     PL05      MC023      OP0190  Morning   
54      PR0000055      2026-07-31     PL01      MC005      OP0212  Morning   
55      PR0000056      2026-09-01     PL06      MC029      OP0239    Night   
63      PR0000064      2026-10-21     PL05      MC022      OP0267  Evening   
69      PR0000070      2026-08-29     PL04      MC017      OP0460  Morning   
86      PR0000087      2026-01-02     PL03      MC014      OP0488  Evening   
109     PR0000110      2026-07-07     PL02      MC008      OP0598  Evening   

    product_code product_family  units_planned  units_produced  \
14       MTR-300         Motors           1086            1042   
16       

In [41]:
import pandas as pd
plant_pdf = pd.read_csv("/content/plant_master.csv")
print("Plant master columns:")
print(plant_pdf.columns)
print(plant_pdf.head())

Plant master columns:
Index(['plant_id', 'plant_name', 'city', 'region', 'year_opened',
       'plant_manager'],
      dtype='object')
  plant_id       plant_name       city region  year_opened plant_manager
0     PL01  Hyderabad Plant  Hyderabad  South         2016      R. Menon
1     PL02       Pune Plant       Pune   West         2018   S. Kulkarni
2     PL03    Chennai Plant    Chennai  South         2015  A. Natarajan
3     PL04      Noida Plant      Noida  North         2019     M. Sharma
4     PL05  Ahmedabad Plant  Ahmedabad   West         2017      P. Desai


In [42]:
%pip -q install "dask[dataframe]"

In [47]:
import dask.dataframe as dd

ddf = dd.read_csv("/content/manufacturing_production_120k.csv",dtype=str)


print(ddf.head())

  production_id production_date plant_id machine_id operator_id    shift  \
0     PR0000001      2026-08-07     PL03      MC011      OP0275  Evening   
1     PR0000002      2026-08-03     PL04      MC019      OP0489  Morning   
2     PR0000003      2026-05-20     PL01      MC002      OP0189  Evening   
3     PR0000004      2026-01-15     PL03      MC013      OP0076  Morning   
4     PR0000005      2026-09-21     PL04      MC020      OP0501  Morning   

  product_code product_family units_planned units_produced defective_units  \
0      MTR-100         Motors          1060            993              30   
1      CTL-330    Controllers           809            836               0   
2      MTR-300         Motors           442            380              17   
3      PMP-210          Pumps           787            720               7   
4      PMP-110          Pumps           344            296               7   

  downtime_minutes energy_kwh temperature_c vibration_mm_s quality_status 

In [48]:
print(ddf.head(5))

  production_id production_date plant_id machine_id operator_id    shift  \
0     PR0000001      2026-08-07     PL03      MC011      OP0275  Evening   
1     PR0000002      2026-08-03     PL04      MC019      OP0489  Morning   
2     PR0000003      2026-05-20     PL01      MC002      OP0189  Evening   
3     PR0000004      2026-01-15     PL03      MC013      OP0076  Morning   
4     PR0000005      2026-09-21     PL04      MC020      OP0501  Morning   

  product_code product_family units_planned units_produced defective_units  \
0      MTR-100         Motors          1060            993              30   
1      CTL-330    Controllers           809            836               0   
2      MTR-300         Motors           442            380              17   
3      PMP-210          Pumps           787            720               7   
4      PMP-110          Pumps           344            296               7   

  downtime_minutes energy_kwh temperature_c vibration_mm_s quality_status 

In [49]:
print("Number of partitions:", ddf.npartitions)

Number of partitions: 1


In [50]:
selected_ddf = ddf[["plant_id", "product_family", "units_produced"]]

print(selected_ddf.head(10))

  plant_id product_family units_produced
0     PL03         Motors            993
1     PL04    Controllers            836
2     PL01         Motors            380
3     PL03          Pumps            720
4     PL04          Pumps            296
5     PL01         Motors            795
6     PL02         Valves            955
7     PL02         Valves            895
8     PL03    Controllers            289
9     PL02    Controllers            527


In [51]:
rejected_ddf = ddf[ddf["quality_status"] == "Reject"]

print(rejected_ddf.head(10))

   production_id production_date plant_id machine_id operator_id    shift  \
1      PR0000002      2026-08-03     PL04      MC019      OP0489  Morning   
2      PR0000003      2026-05-20     PL01      MC002      OP0189  Evening   
7      PR0000008      2026-08-17     PL02      MC009      OP0106    Night   
9      PR0000010      2026-02-13     PL02      MC006      OP0593  Evening   
21     PR0000022      2026-03-03     PL02      MC009      OP0294  Evening   
22     PR0000023      2026-10-15     PL04      MC019      OP0187  Morning   
23     PR0000024      2026-06-07     PL03      MC013      OP0456  Morning   
28     PR0000029      2026-04-30     PL05      MC023      OP0339    Night   
30     PR0000031      2026-01-04     PL01      MC003      OP0524  Evening   
45     PR0000046      2026-04-02     PL06      MC027      OP0484  Morning   

   product_code product_family units_planned units_produced defective_units  \
1       CTL-330    Controllers           809            836              

In [55]:
total_defects_by_family = (ddf.groupby("product_family")["defective_units"].sum())
defects_result = total_defects_by_family.compute()
print(defects_result)

product_family
Controllers    0826232702811270182710160162333264349182773296...
Motors         3017504182332070015613501722822133211020141923...
Pumps          7714132265492326417104921424130412922547212115...
Valves         1051616293825180725193062201611155713172533323...
Name: defective_units, dtype: string


In [58]:
import pandas as pd
import dask.dataframe as dd



pdf = pd.read_csv("/content/manufacturing_production_120k.csv")

pandas_result = ( pdf.groupby("plant_id")["units_produced"] .mean())

print("Pandas Result:")
print(pandas_result)




Pandas Result:
plant_id
PL01    672.998304
PL02    672.781326
PL03    675.665503
PL04    673.835101
PL05    673.124994
PL06    678.006884
Name: units_produced, dtype: float64


In [60]:
from pyspark.sql.functions import col
df_efficiency = df.withColumn(
    "production_efficiency",
    (col("units_produced") / col("units_planned")) * 100)
df_efficiency.select(
    "production_id",
    "units_planned",
    "units_produced",
    "production_efficiency").show(10, truncate=False)

+-------------+-------------+--------------+---------------------+
|production_id|units_planned|units_produced|production_efficiency|
+-------------+-------------+--------------+---------------------+
|PR0000001    |1060         |993           |93.67924528301886    |
|PR0000002    |809          |836           |103.33745364647713   |
|PR0000003    |442          |380           |85.97285067873304    |
|PR0000004    |787          |720           |91.4866581956798     |
|PR0000005    |344          |296           |86.04651162790698    |
|PR0000006    |958          |795           |82.98538622129436    |
|PR0000007    |1165         |955           |81.97424892703863    |
|PR0000008    |862          |895           |103.82830626450117   |
|PR0000009    |313          |289           |92.33226837060703    |
|PR0000010    |532          |527           |99.06015037593986    |
+-------------+-------------+--------------+---------------------+
only showing top 10 rows


In [63]:
df_good = df.withColumn(
    "good_units",
    col("units_produced") - col("defective_units"))
df_good.select(
    "production_id",
    "units_produced",
    "defective_units",
    "good_units").show(10, truncate=False)

+-------------+--------------+---------------+----------+
|production_id|units_produced|defective_units|good_units|
+-------------+--------------+---------------+----------+
|PR0000001    |993           |30             |963       |
|PR0000002    |836           |0              |836       |
|PR0000003    |380           |17             |363       |
|PR0000004    |720           |7              |713       |
|PR0000005    |296           |7              |289       |
|PR0000006    |795           |50             |745       |
|PR0000007    |955           |10             |945       |
|PR0000008    |895           |51             |844       |
|PR0000009    |289           |8              |281       |
|PR0000010    |527           |26             |501       |
+-------------+--------------+---------------+----------+
only showing top 10 rows


In [64]:
from pyspark.sql.functions import lit
df_country = df.withColumn("country",lit("India"))
df_country.select(
    "production_id",
    "plant_id",
    "country").show(10, truncate=False)

+-------------+--------+-------+
|production_id|plant_id|country|
+-------------+--------+-------+
|PR0000001    |PL03    |India  |
|PR0000002    |PL04    |India  |
|PR0000003    |PL01    |India  |
|PR0000004    |PL03    |India  |
|PR0000005    |PL04    |India  |
|PR0000006    |PL01    |India  |
|PR0000007    |PL02    |India  |
|PR0000008    |PL02    |India  |
|PR0000009    |PL03    |India  |
|PR0000010    |PL02    |India  |
+-------------+--------+-------+
only showing top 10 rows


In [65]:
df_renamed = df.withColumnRenamed(
    "energy_kwh",
    "energy_consumed")
df_renamed.select("production_id","energy_consumed").show(10, truncate=False)

+-------------+---------------+
|production_id|energy_consumed|
+-------------+---------------+
|PR0000001    |388.11         |
|PR0000002    |398.58         |
|PR0000003    |183.34         |
|PR0000004    |217.61         |
|PR0000005    |115.47         |
|PR0000006    |290.26         |
|PR0000007    |355.56         |
|PR0000008    |266.06         |
|PR0000009    |87.62          |
|PR0000010    |112.17         |
+-------------+---------------+
only showing top 10 rows


In [66]:
df_double = df.withColumn(
    "units_produced",
    col("units_produced").cast("double"))
df_double.select(
    "production_id",
    "units_produced").show(10, truncate=False)
df_double.printSchema()

+-------------+--------------+
|production_id|units_produced|
+-------------+--------------+
|PR0000001    |993.0         |
|PR0000002    |836.0         |
|PR0000003    |380.0         |
|PR0000004    |720.0         |
|PR0000005    |296.0         |
|PR0000006    |795.0         |
|PR0000007    |955.0         |
|PR0000008    |895.0         |
|PR0000009    |289.0         |
|PR0000010    |527.0         |
+-------------+--------------+
only showing top 10 rows
root
 |-- production_id: string (nullable = true)
 |-- production_date: date (nullable = true)
 |-- plant_id: string (nullable = true)
 |-- machine_id: string (nullable = true)
 |-- operator_id: string (nullable = true)
 |-- shift: string (nullable = true)
 |-- product_code: string (nullable = true)
 |-- product_family: string (nullable = true)
 |-- units_planned: integer (nullable = true)
 |-- units_produced: double (nullable = true)
 |-- defective_units: integer (nullable = true)
 |-- downtime_minutes: integer (nullable = true)
 |-- 

In [73]:
df_defects = df.withColumn(
    "defect_percentage",
    (col("defective_units") / col("units_produced")) * 100)
df_defects.select(
    "production_id",
    "units_produced",
    "defective_units",
    "defect_percentage").show(10, truncate=False)

+-------------+--------------+---------------+------------------+
|production_id|units_produced|defective_units|defect_percentage |
+-------------+--------------+---------------+------------------+
|PR0000001    |993           |30             |3.0211480362537766|
|PR0000002    |836           |0              |0.0               |
|PR0000003    |380           |17             |4.473684210526316 |
|PR0000004    |720           |7              |0.9722222222222222|
|PR0000005    |296           |7              |2.364864864864865 |
|PR0000006    |795           |50             |6.289308176100629 |
|PR0000007    |955           |10             |1.0471204188481675|
|PR0000008    |895           |51             |5.698324022346369 |
|PR0000009    |289           |8              |2.768166089965398 |
|PR0000010    |527           |26             |4.933586337760911 |
+-------------+--------------+---------------+------------------+
only showing top 10 rows


In [75]:
df_energy = df.withColumn(
    "energy_per_unit",
    col("energy_kwh") / col("units_produced"))
df_energy.select(
    "production_id",
    "energy_kwh",
    "units_produced",
    "energy_per_unit").show(10, truncate=False)

+-------------+----------+--------------+-------------------+
|production_id|energy_kwh|units_produced|energy_per_unit    |
+-------------+----------+--------------+-------------------+
|PR0000001    |388.11    |993           |0.39084592145015107|
|PR0000002    |398.58    |836           |0.4767703349282296 |
|PR0000003    |183.34    |380           |0.48247368421052633|
|PR0000004    |217.61    |720           |0.30223611111111115|
|PR0000005    |115.47    |296           |0.3901013513513513 |
|PR0000006    |290.26    |795           |0.3651069182389937 |
|PR0000007    |355.56    |955           |0.37231413612565445|
|PR0000008    |266.06    |895           |0.2972737430167598 |
|PR0000009    |87.62     |289           |0.3031833910034602 |
|PR0000010    |112.17    |527           |0.2128462998102467 |
+-------------+----------+--------------+-------------------+
only showing top 10 rows


In [77]:
from pyspark.sql.functions import col, lit

df_transformed = (
    df
    .withColumn(
        "production_efficiency",
        (col("units_produced") / col("units_planned")) * 100)
    .withColumn(
        "good_units",
        col("units_produced") - col("defective_units"))
    .withColumn("country",lit("India"))
    .withColumnRenamed(
        "energy_kwh",
        "energy_consumed")
    .withColumn(
        "units_produced",
        col("units_produced").cast("double"))
    .withColumn(
        "defect_percentage",
        (col("defective_units") / col("units_produced")) * 100)
    .withColumn(
        "energy_per_unit",
        col("energy_consumed") / col("units_produced")))

final_df = df_transformed.select(
    "production_id",
    "production_efficiency",
    "good_units",
    "country",
    "energy_consumed",
    "units_produced",
    "defect_percentage",
    "energy_per_unit")

final_df.show(10, truncate=False)

final_df.printSchema()

+-------------+---------------------+----------+-------+---------------+--------------+------------------+-------------------+
|production_id|production_efficiency|good_units|country|energy_consumed|units_produced|defect_percentage |energy_per_unit    |
+-------------+---------------------+----------+-------+---------------+--------------+------------------+-------------------+
|PR0000001    |93.67924528301886    |963       |India  |388.11         |993.0         |3.0211480362537766|0.39084592145015107|
|PR0000002    |103.33745364647713   |836       |India  |398.58         |836.0         |0.0               |0.4767703349282296 |
|PR0000003    |85.97285067873304    |363       |India  |183.34         |380.0         |4.473684210526316 |0.48247368421052633|
|PR0000004    |91.4866581956798     |713       |India  |217.61         |720.0         |0.9722222222222222|0.30223611111111115|
|PR0000005    |86.04651162790698    |289       |India  |115.47         |296.0         |2.364864864864865 |0.390

In [81]:
df_no_supervisor = df.drop("supervisor")
print(df_no_supervisor.columns)
df_no_supervisor.show(5, truncate=False)

['production_id', 'production_date', 'plant_id', 'machine_id', 'operator_id', 'shift', 'product_code', 'product_family', 'units_planned', 'units_produced', 'defective_units', 'downtime_minutes', 'energy_kwh', 'temperature_c', 'vibration_mm_s', 'quality_status', 'maintenance_flag']
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+
|production_id|production_date|plant_id|machine_id|operator_id|shift  |product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+
|PR0000001    |2

In [83]:
df_clean_columns = df.drop(
    "supervisor",
    "maintenance_flag")
print(df_clean_columns.columns)
df_clean_columns.show(5, truncate=False)

['production_id', 'production_date', 'plant_id', 'machine_id', 'operator_id', 'shift', 'product_code', 'product_family', 'units_planned', 'units_produced', 'defective_units', 'downtime_minutes', 'energy_kwh', 'temperature_c', 'vibration_mm_s', 'quality_status']
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+
|production_id|production_date|plant_id|machine_id|operator_id|shift  |product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+
|PR0000001    |2026-08-07     |PL03    |MC011     |OP0275     |Evening|MTR-100     |Mot

In [84]:
print("Records before removing duplicates:", df.count())

df_no_duplicates = df.dropDuplicates()

print("Records after removing duplicates:", df_no_duplicates.count())
df_no_duplicates.show(5, truncate=False)

Records before removing duplicates: 120000
Records after removing duplicates: 120000
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|shift  |product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|PR0000304    |2026-07-31     |PL01    |MC004     |OP0518     |Evening|MTR-200     |Motors        |339          |288           |0              |57              |83.44     |58.53   

In [85]:
print("Records before:", df.count())
df_unique_production = df.dropDuplicates(
    ["production_id"])

print("Records after:", df_unique_production.count())

df_unique_production.show(10, truncate=False)

Records before: 120000
Records after: 120000
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|shift  |product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|PR0000002    |2026-08-03     |PL04    |MC019     |OP0489     |Morning|CTL-330     |Controllers   |809          |836           |0              |0               |398.58    |64.37        |2.28          |Reject        |No  

In [86]:
from pyspark.sql.functions import col
print("Records before:", df.count())
df_valid_operator = df.filter(
    col("operator_id").isNotNull())

print("Records after:", df_valid_operator.count())

df_valid_operator.show(10, truncate=False)

Records before: 120000
Records after: 118476
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|shift  |product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|PR0000001    |2026-08-07     |PL03    |MC011     |OP0275     |Evening|MTR-100     |Motors        |1060         |993           |30             |34              |388.11    |61.4         |3.88          |Rework        |No  

In [87]:
unique_plant_machines = df.select("plant_id","machine_id").distinct()

unique_plant_machines.show(truncate=False)

+--------+----------+
|plant_id|machine_id|
+--------+----------+
|PL01    |MC001     |
|PL01    |MC004     |
|PL04    |MC019     |
|PL04    |MC017     |
|PL05    |MC022     |
|PL04    |MC018     |
|PL01    |MC003     |
|PL06    |MC029     |
|PL02    |MC007     |
|PL03    |MC013     |
|PL06    |MC030     |
|PL04    |MC020     |
|PL05    |MC021     |
|PL01    |MC005     |
|PL02    |MC010     |
|PL04    |MC016     |
|PL06    |MC027     |
|PL02    |MC006     |
|PL01    |MC002     |
|PL06    |MC028     |
+--------+----------+
only showing top 20 rows


In [92]:
unique_families = df.select("product_family").distinct()

unique_families.show(truncate=False)

+--------------+
|product_family|
+--------------+
|Pumps         |
|Valves        |
|Controllers   |
|Motors        |
+--------------+



In [93]:
from pyspark.sql.functions import col

df_ascending = df.orderBy(col("units_produced").asc())

df_ascending.show(10, truncate=False)

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|shift  |product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|PR0038643    |2026-08-06     |PL02    |MC009     |OP0361     |Evening|MTR-100     |Motors        |250          |205           |6              |73              |65.18     |55.99        |2.72          |Pass          |No              |Harini    |
|PR0059668    |2026-

In [95]:
df_descending = df.orderBy( col("units_produced").desc())

df_descending.show(10, truncate=False)

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|shift  |product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|PR0087560    |2026-06-29     |PL04    |MC018     |OP0587     |Night  |VLV-120     |Valves        |1199         |1245          |60             |0               |500.87    |58.11        |3.58          |Pass          |Yes             |Charan    |
|PR0008535    |2026-

In [96]:
df_high_defects = df.orderBy(col("defective_units").desc())
df_high_defects.select(
    "production_id",
    "plant_id",
    "product_family",
    "units_produced",
    "defective_units").show(10, truncate=False)

+-------------+--------+--------------+--------------+---------------+
|production_id|plant_id|product_family|units_produced|defective_units|
+-------------+--------+--------------+--------------+---------------+
|PR0051359    |PL02    |Controllers   |1230          |108            |
|PR0108009    |PL05    |Controllers   |1067          |106            |
|PR0067365    |PL01    |Motors        |1186          |103            |
|PR0070440    |PL04    |Valves        |1124          |102            |
|PR0082514    |PL04    |Pumps         |1109          |100            |
|PR0106744    |PL01    |Controllers   |1103          |100            |
|PR0013103    |PL02    |Motors        |990           |97             |
|PR0045241    |PL03    |Valves        |1231          |96             |
|PR0103867    |PL04    |Pumps         |1095          |95             |
|PR0043427    |PL04    |Valves        |1196          |95             |
+-------------+--------+--------------+--------------+---------------+
only s

In [98]:
df_plant_sorted = df.orderBy(col("plant_id").asc(),col("units_produced").desc())
df_plant_sorted.select(
    "production_id",
    "plant_id",
    "units_produced").show(20, truncate=False)

+-------------+--------+--------------+
|production_id|plant_id|units_produced|
+-------------+--------+--------------+
|PR0073059    |PL01    |1239          |
|PR0002678    |PL01    |1238          |
|PR0005192    |PL01    |1236          |
|PR0044516    |PL01    |1236          |
|PR0056319    |PL01    |1236          |
|PR0006350    |PL01    |1234          |
|PR0046906    |PL01    |1233          |
|PR0046582    |PL01    |1230          |
|PR0097172    |PL01    |1229          |
|PR0024941    |PL01    |1229          |
|PR0088933    |PL01    |1229          |
|PR0048075    |PL01    |1228          |
|PR0083644    |PL01    |1228          |
|PR0002284    |PL01    |1227          |
|PR0109962    |PL01    |1227          |
|PR0012821    |PL01    |1226          |
|PR0119269    |PL01    |1226          |
|PR0000300    |PL01    |1226          |
|PR0083683    |PL01    |1226          |
|PR0022084    |PL01    |1226          |
+-------------+--------+--------------+
only showing top 20 rows


In [99]:
top_20_production = (df.orderBy(col("units_produced").desc()).limit(20))

top_20_production.select(
    "production_id",
    "plant_id",
    "machine_id",
    "product_family",
    "units_produced").show(20, truncate=False)

+-------------+--------+----------+--------------+--------------+
|production_id|plant_id|machine_id|product_family|units_produced|
+-------------+--------+----------+--------------+--------------+
|PR0087560    |PL04    |MC018     |Valves        |1245          |
|PR0008535    |PL04    |MC018     |Motors        |1245          |
|PR0098612    |PL05    |MC024     |Pumps         |1245          |
|PR0100473    |PL02    |MC009     |Controllers   |1245          |
|PR0010732    |PL02    |MC007     |Pumps         |1243          |
|PR0082579    |PL03    |MC013     |Motors        |1243          |
|PR0047939    |PL04    |MC020     |Pumps         |1243          |
|PR0001765    |PL02    |MC006     |Motors        |1242          |
|PR0069661    |PL04    |MC020     |Controllers   |1242          |
|PR0064142    |PL06    |MC028     |Pumps         |1241          |
|PR0116814    |PL05    |MC022     |Controllers   |1240          |
|PR0030298    |PL03    |MC011     |Controllers   |1239          |
|PR0093782

In [102]:
top_10_downtime = ( df.orderBy(col("downtime_minutes").desc() ).limit(10))

top_10_downtime.select(
    "production_id",
    "plant_id",
    "machine_id",
    "downtime_minutes",
    "maintenance_flag").show(10, truncate=False)

+-------------+--------+----------+----------------+----------------+
|production_id|plant_id|machine_id|downtime_minutes|maintenance_flag|
+-------------+--------+----------+----------------+----------------+
|PR0092818    |PL06    |MC027     |281             |No              |
|PR0045363    |PL02    |MC010     |278             |No              |
|PR0020842    |PL02    |MC008     |267             |No              |
|PR0065023    |PL06    |MC029     |267             |No              |
|PR0068768    |PL03    |MC012     |267             |No              |
|PR0100950    |PL03    |MC012     |263             |No              |
|PR0041129    |PL01    |MC005     |263             |No              |
|PR0049997    |PL04    |MC019     |263             |Yes             |
|PR0022495    |PL05    |MC022     |262             |No              |
|PR0112855    |PL02    |MC008     |261             |Yes             |
+-------------+--------+----------+----------------+----------------+



In [108]:
from pyspark.sql.functions import sum, avg, max, min, count

In [109]:
from pyspark.sql.functions import sum

df.select(sum("units_planned").alias("total_units_planned")).show()

+-------------------+
|total_units_planned|
+-------------------+
|           87094621|
+-------------------+



In [111]:
df.select(sum("units_produced").alias("total_units_produced")).show()

+--------------------+
|total_units_produced|
+--------------------+
|            80927888|
+--------------------+



In [114]:
df.select( sum("defective_units").alias("total_defective_units")).show()

+---------------------+
|total_defective_units|
+---------------------+
|              2030244|
+---------------------+



In [116]:
from pyspark.sql.functions import avg
df.select( avg("units_produced").alias("average_units_produced")).show()

+----------------------+
|average_units_produced|
+----------------------+
|     674.3990666666666|
+----------------------+



In [118]:
from pyspark.sql.functions import max

df.select(max("units_produced").alias("maximum_production_output")).show()

+-------------------------+
|maximum_production_output|
+-------------------------+
|                     1245|
+-------------------------+



In [120]:
from pyspark.sql.functions import min

df.select( min("units_produced").alias("minimum_production_output")).show()

+-------------------------+
|minimum_production_output|
+-------------------------+
|                      205|
+-------------------------+



In [123]:
df.select(avg("downtime_minutes").alias("average_downtime_minutes")).show()

+------------------------+
|average_downtime_minutes|
+------------------------+
|       33.93149474428351|
+------------------------+



In [125]:
df.select(sum("energy_kwh").alias("total_energy_kwh")).show()

+--------------------+
|    total_energy_kwh|
+--------------------+
|2.8368014080000117E7|
+--------------------+



In [126]:
df.select(avg("temperature_c").alias("average_temperature_c")).show()

+---------------------+
|average_temperature_c|
+---------------------+
|   63.021532392167266|
+---------------------+



In [128]:
df.select(avg("vibration_mm_s").alias("average_vibration_mm_s")).show()

+----------------------+
|average_vibration_mm_s|
+----------------------+
|     3.398737691604917|
+----------------------+



In [130]:
from pyspark.sql.functions import sum
production_by_plant_family = df.groupBy(
    "plant_id",
    "product_family").agg(sum("units_produced").alias("total_production"))
production_by_plant_family.show(truncate=False)

+--------+--------------+----------------+
|plant_id|product_family|total_production|
+--------+--------------+----------------+
|PL06    |Controllers   |3415174         |
|PL03    |Valves        |3413649         |
|PL02    |Pumps         |3379628         |
|PL05    |Pumps         |3359859         |
|PL03    |Pumps         |3378880         |
|PL01    |Motors        |3304675         |
|PL05    |Valves        |3305648         |
|PL05    |Controllers   |3302303         |
|PL04    |Pumps         |3324461         |
|PL04    |Valves        |3465316         |
|PL02    |Controllers   |3399859         |
|PL02    |Valves        |3401146         |
|PL01    |Controllers   |3316184         |
|PL01    |Pumps         |3421797         |
|PL06    |Pumps         |3344114         |
|PL06    |Motors        |3331302         |
|PL04    |Controllers   |3347255         |
|PL02    |Motors        |3365819         |
|PL03    |Motors        |3367443         |
|PL05    |Motors        |3355353         |
+--------+-

In [131]:
defects_by_plant = df.groupBy("plant_id").agg(sum("defective_units").alias("total_defective_units"))

defects_by_plant.show(truncate=False)

+--------+---------------------+
|plant_id|total_defective_units|
+--------+---------------------+
|PL04    |338118               |
|PL02    |339647               |
|PL06    |338993               |
|PL05    |334502               |
|PL03    |339264               |
|PL01    |339720               |
+--------+---------------------+



In [133]:
from pyspark.sql.functions import avg
downtime_by_shift = df.groupBy("shift").agg(avg("downtime_minutes").alias("average_downtime_minutes"))
downtime_by_shift.show(truncate=False)

+-------+------------------------+
|shift  |average_downtime_minutes|
+-------+------------------------+
|Evening|34.2213017151833        |
|Morning|33.61973080530278       |
|Night  |33.953058820580935      |
+-------+------------------------+



In [134]:
avg_energy_by_family = df.groupBy(
    "product_family").agg(
    avg("energy_kwh").alias("average_energy_kwh"))

avg_energy_by_family.show(truncate=False)

+--------------+------------------+
|product_family|average_energy_kwh|
+--------------+------------------+
|Pumps         |236.89137927582252|
|Valves        |236.34508750495314|
|Controllers   |235.15370433333288|
|Motors        |237.21818224597516|
+--------------+------------------+



In [135]:
production_by_quality = df.groupBy(
    "quality_status").agg(
    count("*").alias("production_record_count"))

production_by_quality.show(truncate=False)

+--------------+-----------------------+
|quality_status|production_record_count|
+--------------+-----------------------+
|Rework        |22928                  |
|Reject        |22744                  |
|Pass          |74328                  |
+--------------+-----------------------+



In [136]:
production_by_plant_shift = df.groupBy(
    "plant_id",
    "shift").agg(
    sum("units_produced").alias("total_units_produced"))

production_by_plant_shift.show(truncate=False)

+--------+-------+--------------------+
|plant_id|shift  |total_units_produced|
+--------+-------+--------------------+
|PL02    |Morning|4561068             |
|PL03    |Night  |4599608             |
|PL04    |Night  |4492219             |
|PL01    |Morning|4447851             |
|PL04    |Evening|4540617             |
|PL05    |Morning|4431446             |
|PL02    |Night  |4558467             |
|PL01    |Night  |4562612             |
|PL05    |Evening|4475101             |
|PL05    |Night  |4416616             |
|PL03    |Morning|4460950             |
|PL03    |Evening|4487211             |
|PL06    |Evening|4458798             |
|PL06    |Night  |4604432             |
|PL01    |Evening|4477769             |
|PL02    |Evening|4426917             |
|PL06    |Morning|4429107             |
|PL04    |Morning|4497099             |
+--------+-------+--------------------+



In [137]:
defects_by_plant_family = df.groupBy(
    "plant_id",
    "product_family").agg(
    sum("defective_units").alias("total_defective_units"))

defects_by_plant_family.show(truncate=False)

+--------+--------------+---------------------+
|plant_id|product_family|total_defective_units|
+--------+--------------+---------------------+
|PL06    |Controllers   |86140                |
|PL03    |Valves        |85333                |
|PL02    |Pumps         |84491                |
|PL05    |Pumps         |83607                |
|PL03    |Pumps         |84803                |
|PL01    |Motors        |84174                |
|PL05    |Valves        |82975                |
|PL05    |Controllers   |83589                |
|PL04    |Pumps         |83976                |
|PL04    |Valves        |87084                |
|PL02    |Controllers   |87120                |
|PL02    |Valves        |83476                |
|PL01    |Controllers   |83569                |
|PL01    |Pumps         |87103                |
|PL06    |Pumps         |83884                |
|PL06    |Motors        |83399                |
|PL04    |Controllers   |83161                |
|PL02    |Motors        |84560          

In [138]:
avg_temperature_by_machine = df.groupBy("machine_id").agg(avg("temperature_c").alias("average_temperature_c"))

avg_temperature_by_machine.show(truncate=False)

+----------+---------------------+
|machine_id|average_temperature_c|
+----------+---------------------+
|MC003     |62.879653934888516   |
|MC018     |62.81540393578454    |
|MC017     |63.07674442190667    |
|MC009     |63.1252779855396     |
|MC014     |63.114309957574186   |
|MC007     |63.037661716587      |
|MC002     |62.93530761472517    |
|MC024     |62.96457946271644    |
|MC016     |63.03639870582378    |
|MC026     |63.02672521957329    |
|MC005     |62.979236446933044   |
|MC010     |62.883106870229064   |
|MC012     |62.95193507800698    |
|MC023     |63.109051746112684   |
|MC021     |63.01464069491981    |
|MC020     |63.22489208633087    |
|MC011     |63.060230414746606   |
|MC013     |62.959448777951046   |
|MC025     |62.898440510283834   |
|MC028     |62.97779683034558    |
+----------+---------------------+
only showing top 20 rows


In [140]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, avg


plant_df = spark.read.csv(
    "/content/plant_master.csv",
    header=True,
    inferSchema=True)


machine_df = spark.read.csv(
    "/content/machine_master.csv",
    header=True,
    inferSchema=True)

print("Production Data:")
df.show(5)

print("Plant Master Data:")
plant_df.show(5)

print("Machine Master Data:")
machine_df.show(5)

Production Data:
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|    PR0000001|     2026-08-07|    PL03|     MC011|     OP0275|Evening|     MTR-100|        Motors|         1060|           993|             30|              34|    388.11|         61.4|          3.88|        Rework|              No|    Charan|
|  

In [141]:
production_plant = df.join(
    plant_df,
    on="plant_id",
    how="inner")
production_plant.show(10, truncate=False)

+--------+-------------+---------------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+---------------+---------+------+-----------+-------------+
|plant_id|production_id|production_date|machine_id|operator_id|shift  |product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|plant_name     |city     |region|year_opened|plant_manager|
+--------+-------------+---------------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+---------------+---------+------+-----------+-------------+
|PL03    |PR0000001    |2026-08-07     |MC011     |OP0275     |Evening|MTR-100     |Moto

In [143]:
production_plant.select(
    col("production_id"),
    col("plant_id"),
    col("plant_name"),
    col("city"),
    col("units_produced")).show(10, truncate=False)

+-------------+--------+---------------+---------+--------------+
|production_id|plant_id|plant_name     |city     |units_produced|
+-------------+--------+---------------+---------+--------------+
|PR0000001    |PL03    |Chennai Plant  |Chennai  |993           |
|PR0000002    |PL04    |Noida Plant    |Noida    |836           |
|PR0000003    |PL01    |Hyderabad Plant|Hyderabad|380           |
|PR0000004    |PL03    |Chennai Plant  |Chennai  |720           |
|PR0000005    |PL04    |Noida Plant    |Noida    |296           |
|PR0000006    |PL01    |Hyderabad Plant|Hyderabad|795           |
|PR0000007    |PL02    |Pune Plant     |Pune     |955           |
|PR0000008    |PL02    |Pune Plant     |Pune     |895           |
|PR0000009    |PL03    |Chennai Plant  |Chennai  |289           |
|PR0000010    |PL02    |Pune Plant     |Pune     |527           |
+-------------+--------+---------------+---------+--------------+
only showing top 10 rows


In [145]:
production_machine = df.join(
    machine_df,
    on="machine_id",
    how="inner")
production_machine.show(10, truncate=False)

+----------+-------------+---------------+--------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+--------+------------+-----------------+--------------+
|machine_id|production_id|production_date|plant_id|operator_id|shift  |product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|plant_id|machine_type|capacity_per_hour|installed_year|
+----------+-------------+---------------+--------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+--------+------------+-----------------+--------------+
|MC011     |PR0000001    |2026-08-07     |PL03    |OP0275     |Evening|MTR-100     |Motors        |1

In [146]:
complete_df = df.join(
    plant_df,
    on="plant_id",
    how="inner").join(
    machine_df,
    on="machine_id",
    how="inner")
complete_df.show(10, truncate=False)

+----------+--------+-------------+---------------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+---------------+---------+------+-----------+-------------+--------+------------+-----------------+--------------+
|machine_id|plant_id|production_id|production_date|operator_id|shift  |product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|plant_name     |city     |region|year_opened|plant_manager|plant_id|machine_type|capacity_per_hour|installed_year|
+----------+--------+-------------+---------------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+---------------+---------+------+-----

In [147]:
city_production = df.join(
    plant_df,
    on="plant_id",
    how="inner").groupBy(
    "city").agg(
    sum("units_produced").alias("total_production")).orderBy(
    col("total_production").desc())

city_production.show(truncate=False)

+---------+----------------+
|city     |total_production|
+---------+----------------+
|Chennai  |13547769        |
|Pune     |13546452        |
|Noida    |13529935        |
|Kolkata  |13492337        |
|Hyderabad|13488232        |
|Ahmedabad|13323163        |
+---------+----------------+



In [149]:
machine_downtime = df.join(
    machine_df,
    on="machine_id",
    how="inner").groupBy(
    "machine_type").agg(
    avg("downtime_minutes").alias("average_downtime")).orderBy(
    col("average_downtime").desc())

machine_downtime.show(truncate=False)

+------------+-----------------+
|machine_type|average_downtime |
+------------+-----------------+
|Casting     |34.1660783396419 |
|Welding     |34.00449382099613|
|Assembly    |33.96718284443325|
|Testing     |33.87585594724829|
|CNC         |33.64548129466162|
+------------+-----------------+



In [150]:
unmatched_machines = df.join(
    machine_df,
    on="machine_id",
    how="left_anti")

unmatched_machines.show(20, truncate=False)

+----------+-------------+---------------+--------+-----------+-----+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|machine_id|production_id|production_date|plant_id|operator_id|shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+----------+-------------+---------------+--------+-----------+-----+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
+----------+-------------+---------------+--------+-----------+-----+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+



In [151]:
left_join_df = df.join(
    machine_df,
    on="machine_id",
    how="left")

left_join_df.show(10, truncate=False)

+----------+-------------+---------------+--------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+--------+------------+-----------------+--------------+
|machine_id|production_id|production_date|plant_id|operator_id|shift  |product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|plant_id|machine_type|capacity_per_hour|installed_year|
+----------+-------------+---------------+--------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+--------+------------+-----------------+--------------+
|MC011     |PR0000001    |2026-08-07     |PL03    |OP0275     |Evening|MTR-100     |Motors        |1

In [152]:
energy_by_plant = df.join(
    plant_df,
    on="plant_id",
    how="inner").groupBy(
    "plant_name").agg(
    sum("energy_kwh").alias("total_energy_kwh")).orderBy(
    col("total_energy_kwh").desc())

energy_by_plant.show(truncate=False)

+---------------+-----------------+
|plant_name     |total_energy_kwh |
+---------------+-----------------+
|Pune Plant     |4757984.949999988|
|Noida Plant    |4746238.580000002|
|Chennai Plant  |4742748.169999995|
|Hyderabad Plant|4725471.189999996|
|Kolkata Plant  |4723180.160000009|
|Ahmedabad Plant|4672391.03000001 |
+---------------+-----------------+



In [153]:
from pyspark.sql.functions import col, when
df_updated1 = df.withColumn(
    "quality_status",
    when(col("quality_status") == "Reject",
        "Failed").otherwise(
        col("quality_status")))
df_updated1.select(
    "production_id",
    "quality_status").show(10, truncate=False)

+-------------+--------------+
|production_id|quality_status|
+-------------+--------------+
|PR0000001    |Rework        |
|PR0000002    |Failed        |
|PR0000003    |Failed        |
|PR0000004    |Rework        |
|PR0000005    |Pass          |
|PR0000006    |Rework        |
|PR0000007    |Pass          |
|PR0000008    |Failed        |
|PR0000009    |Pass          |
|PR0000010    |Failed        |
+-------------+--------------+
only showing top 10 rows


In [155]:
from pyspark.sql.functions import col, when

df_updated1 = df.withColumn(
    "quality_status",
    when(
        col("quality_status") == "Reject",
        "Failed").otherwise(
        col("quality_status")))

df_updated1.select(
    "production_id",
    "quality_status").show(10, truncate=False)

+-------------+--------------+
|production_id|quality_status|
+-------------+--------------+
|PR0000001    |Rework        |
|PR0000002    |Failed        |
|PR0000003    |Failed        |
|PR0000004    |Rework        |
|PR0000005    |Pass          |
|PR0000006    |Rework        |
|PR0000007    |Pass          |
|PR0000008    |Failed        |
|PR0000009    |Pass          |
|PR0000010    |Failed        |
+-------------+--------------+
only showing top 10 rows


In [157]:
df_updated2 = df.withColumn(
    "quality_status",
    when(
        col("quality_status") == "Rework",
        "Requires Review").otherwise(
        col("quality_status")    ))
df_updated2.select(
    "production_id",
    "quality_status").show(10, truncate=False)

+-------------+---------------+
|production_id|quality_status |
+-------------+---------------+
|PR0000001    |Requires Review|
|PR0000002    |Reject         |
|PR0000003    |Reject         |
|PR0000004    |Requires Review|
|PR0000005    |Pass           |
|PR0000006    |Requires Review|
|PR0000007    |Pass           |
|PR0000008    |Reject         |
|PR0000009    |Pass           |
|PR0000010    |Reject         |
+-------------+---------------+
only showing top 10 rows


In [158]:
df_updated4 = df.withColumn(
    "energy_kwh",
    when(
        col("shift") == "Night",
        col("energy_kwh") * 1.03).otherwise(
        col("energy_kwh")))

df_updated4.select(
    "production_id",
    "shift",
    "energy_kwh").show(10, truncate=False)

+-------------+-------+----------+
|production_id|shift  |energy_kwh|
+-------------+-------+----------+
|PR0000001    |Evening|388.11    |
|PR0000002    |Morning|398.58    |
|PR0000003    |Evening|183.34    |
|PR0000004    |Morning|217.61    |
|PR0000005    |Morning|115.47    |
|PR0000006    |Evening|290.26    |
|PR0000007    |Morning|355.56    |
|PR0000008    |Night  |274.0418  |
|PR0000009    |Evening|87.62     |
|PR0000010    |Evening|112.17    |
+-------------+-------+----------+
only showing top 10 rows


In [159]:
df_updated5 = df.withColumn(
    "operator_id",
    when(
        col("operator_id").isNull(),
        "UNASSIGNED").otherwise(
        col("operator_id")))
df_updated5.select(
    "production_id",
    "operator_id").show(10, truncate=False)

+-------------+-----------+
|production_id|operator_id|
+-------------+-----------+
|PR0000001    |OP0275     |
|PR0000002    |OP0489     |
|PR0000003    |OP0189     |
|PR0000004    |OP0076     |
|PR0000005    |OP0501     |
|PR0000006    |OP0501     |
|PR0000007    |OP0645     |
|PR0000008    |OP0106     |
|PR0000009    |OP0386     |
|PR0000010    |OP0593     |
+-------------+-----------+
only showing top 10 rows


In [160]:
df_updated6 = df.withColumn(
    "supervisor",
    when(
        col("supervisor") == "Anita",
        "Anita Rao").otherwise(
        col("supervisor") ))
df_updated6.select(
    "production_id",
    "supervisor").show(10, truncate=False)

+-------------+----------+
|production_id|supervisor|
+-------------+----------+
|PR0000001    |Charan    |
|PR0000002    |Farah     |
|PR0000003    |Deepa     |
|PR0000004    |Farah     |
|PR0000005    |Deepa     |
|PR0000006    |Gopal     |
|PR0000007    |Gopal     |
|PR0000008    |Irfan     |
|PR0000009    |Anita Rao |
|PR0000010    |Gopal     |
+-------------+----------+
only showing top 10 rows


In [161]:
from pyspark.sql.functions import round

df_updated7 = df.withColumn(
    "corrected_temperature_c",
    round(col("temperature_c"), 1))
df_updated7.select(
    "production_id",
    "temperature_c",
    "corrected_temperature_c").show(10, truncate=False)

+-------------+-------------+-----------------------+
|production_id|temperature_c|corrected_temperature_c|
+-------------+-------------+-----------------------+
|PR0000001    |61.4         |61.4                   |
|PR0000002    |64.37        |64.4                   |
|PR0000003    |54.13        |54.1                   |
|PR0000004    |64.47        |64.5                   |
|PR0000005    |72.59        |72.6                   |
|PR0000006    |64.91        |64.9                   |
|PR0000007    |62.76        |62.8                   |
|PR0000008    |57.1         |57.1                   |
|PR0000009    |66.55        |66.6                   |
|PR0000010    |60.96        |61.0                   |
+-------------+-------------+-----------------------+
only showing top 10 rows


In [162]:
from pyspark.sql.functions import col

missing_operator_count = df.filter(col("operator_id").isNull()).count()

print("Missing operator IDs:", missing_operator_count)

Missing operator IDs: 1524


In [163]:
missing_temperature_count = df.filter(col("temperature_c").isNull()).count()

print("Missing temperature values:", missing_temperature_count)

Missing temperature values: 2132


In [167]:
missing_vibration_count = df.filter(
  col("vibration_mm_s").isNull()).count()

print("Missing vibration values:", missing_vibration_count)

Missing vibration values: 1788


In [169]:
missing_downtime_count = df.filter(
    col("downtime_minutes").isNull()).count()

print("Missing downtime values:", missing_downtime_count)

Missing downtime values: 1177


In [170]:
df_no_nulls = df.na.drop()

print("Original record count:", df.count())
print("Records after dropping NULLs:", df_no_nulls.count())

df_no_nulls.show(10, truncate=False)

Original record count: 120000
Records after dropping NULLs: 113531
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|shift  |product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|PR0000001    |2026-08-07     |PL03    |MC011     |OP0275     |Evening|MTR-100     |Motors        |1060         |993           |30             |34              |388.11    |61.4         |3.88        

In [171]:
df_no_missing_temperature = df.na.drop(
    subset=["temperature_c"])

print("Original record count:", df.count())
print("Records after removing missing temperatures:",
    df_no_missing_temperature.count())

df_no_missing_temperature.show(10, truncate=False)

Original record count: 120000
Records after removing missing temperatures: 117868
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|shift  |product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|PR0000001    |2026-08-07     |PL03    |MC011     |OP0275     |Evening|MTR-100     |Motors        |1060         |993           |30             |34              |388.11    |61.4       

In [173]:
df_filled_downtime = df.fillna({
    "downtime_minutes": 0})

df_filled_downtime.select(
    "production_id",
    "downtime_minutes").show(10, truncate=False)

+-------------+----------------+
|production_id|downtime_minutes|
+-------------+----------------+
|PR0000001    |34              |
|PR0000002    |0               |
|PR0000003    |15              |
|PR0000004    |39              |
|PR0000005    |49              |
|PR0000006    |44              |
|PR0000007    |5               |
|PR0000008    |77              |
|PR0000009    |16              |
|PR0000010    |64              |
+-------------+----------------+
only showing top 10 rows


In [174]:
df_filled_operator = df.fillna({
    "operator_id": "UNKNOWN"})

df_filled_operator.select(
    "production_id",
    "operator_id").show(10, truncate=False)

+-------------+-----------+
|production_id|operator_id|
+-------------+-----------+
|PR0000001    |OP0275     |
|PR0000002    |OP0489     |
|PR0000003    |OP0189     |
|PR0000004    |OP0076     |
|PR0000005    |OP0501     |
|PR0000006    |OP0501     |
|PR0000007    |OP0645     |
|PR0000008    |OP0106     |
|PR0000009    |OP0386     |
|PR0000010    |OP0593     |
+-------------+-----------+
only showing top 10 rows


In [176]:
from pyspark.sql.functions import avg, col

average_temperature = df.select(
    avg("temperature_c").alias("average_temperature")).first()["average_temperature"]

print("Average temperature:", average_temperature)

df_filled_temperature = df.fillna({
    "temperature_c": average_temperature})

df_filled_temperature.select(
    "production_id",
    "temperature_c").show(10, truncate=False)

Average temperature: 63.021532392167266
+-------------+-------------+
|production_id|temperature_c|
+-------------+-------------+
|PR0000001    |61.4         |
|PR0000002    |64.37        |
|PR0000003    |54.13        |
|PR0000004    |64.47        |
|PR0000005    |72.59        |
|PR0000006    |64.91        |
|PR0000007    |62.76        |
|PR0000008    |57.1         |
|PR0000009    |66.55        |
|PR0000010    |60.96        |
+-------------+-------------+
only showing top 10 rows


In [178]:
from pyspark.sql.functions import upper, col

df_upper = df.withColumn(
    "product_family_upper",
    upper(col("product_family")))

df_upper.select(
    "product_family",
    "product_family_upper").show(10, truncate=False)

+--------------+--------------------+
|product_family|product_family_upper|
+--------------+--------------------+
|Motors        |MOTORS              |
|Controllers   |CONTROLLERS         |
|Motors        |MOTORS              |
|Pumps         |PUMPS               |
|Pumps         |PUMPS               |
|Motors        |MOTORS              |
|Valves        |VALVES              |
|Valves        |VALVES              |
|Controllers   |CONTROLLERS         |
|Controllers   |CONTROLLERS         |
+--------------+--------------------+
only showing top 10 rows


In [181]:
from pyspark.sql.functions import lower

df_lower = df.withColumn(
    "supervisor_lower",
    lower(col("supervisor")))

df_lower.select(
    "supervisor",
    "supervisor_lower").show(10, truncate=False)

+----------+----------------+
|supervisor|supervisor_lower|
+----------+----------------+
|Charan    |charan          |
|Farah     |farah           |
|Deepa     |deepa           |
|Farah     |farah           |
|Deepa     |deepa           |
|Gopal     |gopal           |
|Gopal     |gopal           |
|Irfan     |irfan           |
|Anita     |anita           |
|Gopal     |gopal           |
+----------+----------------+
only showing top 10 rows


In [183]:
from pyspark.sql.functions import trim

df_trimmed = df.withColumn(
    "operator_id_trimmed",
    trim(col("operator_id")))

df_trimmed.select(
    "operator_id",
    "operator_id_trimmed").show(10, truncate=False)

+-----------+-------------------+
|operator_id|operator_id_trimmed|
+-----------+-------------------+
|OP0275     |OP0275             |
|OP0489     |OP0489             |
|OP0189     |OP0189             |
|OP0076     |OP0076             |
|OP0501     |OP0501             |
|OP0501     |OP0501             |
|OP0645     |OP0645             |
|OP0106     |OP0106             |
|OP0386     |OP0386             |
|OP0593     |OP0593             |
+-----------+-------------------+
only showing top 10 rows


In [184]:
from pyspark.sql.functions import length

df_product_length = df.withColumn(
    "product_code_length",
    length(col("product_code")))

df_product_length.select(
    "product_code",
    "product_code_length").show(10, truncate=False)

+------------+-------------------+
|product_code|product_code_length|
+------------+-------------------+
|MTR-100     |7                  |
|CTL-330     |7                  |
|MTR-300     |7                  |
|PMP-210     |7                  |
|PMP-110     |7                  |
|MTR-300     |7                  |
|VLV-120     |7                  |
|VLV-320     |7                  |
|CTL-130     |7                  |
|CTL-230     |7                  |
+------------+-------------------+
only showing top 10 rows


In [185]:
from pyspark.sql.functions import concat, lit

df_combined = df.withColumn(
    "plant_machine",
    concat(
        col("plant_id"),
        lit("-"),
        col("machine_id")))

df_combined.select(
    "plant_id",
    "machine_id",
    "plant_machine").show(10, truncate=False)

+--------+----------+-------------+
|plant_id|machine_id|plant_machine|
+--------+----------+-------------+
|PL03    |MC011     |PL03-MC011   |
|PL04    |MC019     |PL04-MC019   |
|PL01    |MC002     |PL01-MC002   |
|PL03    |MC013     |PL03-MC013   |
|PL04    |MC020     |PL04-MC020   |
|PL01    |MC002     |PL01-MC002   |
|PL02    |MC007     |PL02-MC007   |
|PL02    |MC009     |PL02-MC009   |
|PL03    |MC015     |PL03-MC015   |
|PL02    |MC006     |PL02-MC006   |
+--------+----------+-------------+
only showing top 10 rows


In [186]:
from pyspark.sql.functions import round

df_rounded = df.withColumn(
    "temperature_rounded",
    round(col("temperature_c"), 1))

df_rounded.select(
    "temperature_c",
    "temperature_rounded").show(10, truncate=False)

+-------------+-------------------+
|temperature_c|temperature_rounded|
+-------------+-------------------+
|61.4         |61.4               |
|64.37        |64.4               |
|54.13        |54.1               |
|64.47        |64.5               |
|72.59        |72.6               |
|64.91        |64.9               |
|62.76        |62.8               |
|57.1         |57.1               |
|66.55        |66.6               |
|60.96        |61.0               |
+-------------+-------------------+
only showing top 10 rows


In [189]:
from pyspark.sql.functions import abs

df_difference = df.withColumn(
    "production_difference",
    abs(col("units_planned") - col("units_produced")))

df_difference.select(
    "production_id",
    "units_planned",
    "units_produced",
    "production_difference").show(10, truncate=False)

+-------------+-------------+--------------+---------------------+
|production_id|units_planned|units_produced|production_difference|
+-------------+-------------+--------------+---------------------+
|PR0000001    |1060         |993           |67                   |
|PR0000002    |809          |836           |27                   |
|PR0000003    |442          |380           |62                   |
|PR0000004    |787          |720           |67                   |
|PR0000005    |344          |296           |48                   |
|PR0000006    |958          |795           |163                  |
|PR0000007    |1165         |955           |210                  |
|PR0000008    |862          |895           |33                   |
|PR0000009    |313          |289           |24                   |
|PR0000010    |532          |527           |5                    |
+-------------+-------------+--------------+---------------------+
only showing top 10 rows


In [192]:
from pyspark.sql.functions import regexp_replace

df_id_replaced = df.withColumn(
    "production_id_modified",
    regexp_replace( col("production_id"),"-","_"))

df_id_replaced.select(
    "production_id",
    "production_id_modified").show(10, truncate=False)

+-------------+----------------------+
|production_id|production_id_modified|
+-------------+----------------------+
|PR0000001    |PR0000001             |
|PR0000002    |PR0000002             |
|PR0000003    |PR0000003             |
|PR0000004    |PR0000004             |
|PR0000005    |PR0000005             |
|PR0000006    |PR0000006             |
|PR0000007    |PR0000007             |
|PR0000008    |PR0000008             |
|PR0000009    |PR0000009             |
|PR0000010    |PR0000010             |
+-------------+----------------------+
only showing top 10 rows


In [196]:
from pyspark.sql.functions import substring

df_substring = df.withColumn(
    "product_prefix",
    substring(col("product_code"), 1, 3   ))

df_substring.select(
    "product_code",
    "product_prefix").show(10, truncate=False)

+------------+--------------+
|product_code|product_prefix|
+------------+--------------+
|MTR-100     |MTR           |
|CTL-330     |CTL           |
|MTR-300     |MTR           |
|PMP-210     |PMP           |
|PMP-110     |PMP           |
|MTR-300     |MTR           |
|VLV-120     |VLV           |
|VLV-320     |VLV           |
|CTL-130     |CTL           |
|CTL-230     |CTL           |
+------------+--------------+
only showing top 10 rows


In [198]:
from pyspark.sql.functions import col, when, round

df_topic14_q1 = (
    df
    .withColumn(
        "production_efficiency",
        when(
            col("units_planned") > 0,
            (col("units_produced") / col("units_planned")) * 100))
    .withColumn(
        "efficiency_category",
        when(col("production_efficiency") >= 100, "Above Target")
        .when(col("production_efficiency") >= 95, "On Target")
        .when(col("production_efficiency") >= 85, "Below Target")
        .otherwise("Critical") ))

df_topic14_q1.select(
    "production_id",
    "units_planned",
    "units_produced",
    round(col("production_efficiency"), 2).alias("efficiency_percentage"),
    "efficiency_category").show(10, truncate=False)

+-------------+-------------+--------------+---------------------+-------------------+
|production_id|units_planned|units_produced|efficiency_percentage|efficiency_category|
+-------------+-------------+--------------+---------------------+-------------------+
|PR0000001    |1060         |993           |93.68                |Below Target       |
|PR0000002    |809          |836           |103.34               |Above Target       |
|PR0000003    |442          |380           |85.97                |Below Target       |
|PR0000004    |787          |720           |91.49                |Below Target       |
|PR0000005    |344          |296           |86.05                |Below Target       |
|PR0000006    |958          |795           |82.99                |Critical           |
|PR0000007    |1165         |955           |81.97                |Critical           |
|PR0000008    |862          |895           |103.83               |Above Target       |
|PR0000009    |313          |289           

In [199]:
from pyspark.sql.functions import col, when

df_topic14_q2 = df.withColumn(
    "defect_category",
    when(col("defective_units") == 0, "Perfect")
    .when(col("defective_units") <= 10, "Normal")
    .when(col("defective_units") <= 25, "Attention")
    .otherwise("High Defect"))
df_topic14_q2.select(
    "production_id",
    "defective_units",
    "defect_category").show(10, truncate=False)

+-------------+---------------+---------------+
|production_id|defective_units|defect_category|
+-------------+---------------+---------------+
|PR0000001    |30             |High Defect    |
|PR0000002    |0              |Perfect        |
|PR0000003    |17             |Attention      |
|PR0000004    |7              |Normal         |
|PR0000005    |7              |Normal         |
|PR0000006    |50             |High Defect    |
|PR0000007    |10             |Normal         |
|PR0000008    |51             |High Defect    |
|PR0000009    |8              |Normal         |
|PR0000010    |26             |High Defect    |
+-------------+---------------+---------------+
only showing top 10 rows


In [201]:
from pyspark.sql.functions import col, when

TEMP_LIMIT = 75
VIBRATION_LIMIT = 5

df_topic14_q3 = df.withColumn(
    "machine_condition",
    when(
        (col("temperature_c") > TEMP_LIMIT) |
        (col("vibration_mm_s") > VIBRATION_LIMIT),
        "Inspect").otherwise("Normal"))

df_topic14_q3.select(
    "production_id",
    "machine_id",
    "temperature_c",
    "vibration_mm_s",
    "machine_condition").show(10, truncate=False)

+-------------+----------+-------------+--------------+-----------------+
|production_id|machine_id|temperature_c|vibration_mm_s|machine_condition|
+-------------+----------+-------------+--------------+-----------------+
|PR0000001    |MC011     |61.4         |3.88          |Normal           |
|PR0000002    |MC019     |64.37        |2.28          |Normal           |
|PR0000003    |MC002     |54.13        |4.66          |Normal           |
|PR0000004    |MC013     |64.47        |NULL          |Normal           |
|PR0000005    |MC020     |72.59        |3.21          |Normal           |
|PR0000006    |MC002     |64.91        |2.77          |Normal           |
|PR0000007    |MC007     |62.76        |2.89          |Normal           |
|PR0000008    |MC009     |57.1         |1.34          |Normal           |
|PR0000009    |MC015     |66.55        |3.07          |Normal           |
|PR0000010    |MC006     |60.96        |3.48          |Normal           |
+-------------+----------+------------

In [202]:
df_topic14_q1.groupBy(
    "efficiency_category").count().orderBy(
    "efficiency_category").show(truncate=False)

+-------------------+-----+
|efficiency_category|count|
+-------------------+-----+
|Above Target       |21735|
|Below Target       |54619|
|Critical           |16821|
|On Target          |26825|
+-------------------+-----+



In [203]:
df_topic14_q2.groupBy(
    "defect_category").count().orderBy(
    "defect_category").show(truncate=False)

+---------------+-----+
|defect_category|count|
+---------------+-----+
|Attention      |45745|
|High Defect    |28058|
|Normal         |34706|
|Perfect        |11491|
+---------------+-----+



In [204]:
df_topic14_q3.groupBy(
    "machine_condition").count().orderBy(
    "machine_condition").show(truncate=False)

+-----------------+------+
|machine_condition|count |
+-----------------+------+
|Inspect          |15578 |
|Normal           |104422|
+-----------------+------+



In [205]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("ComplexJSON") \
    .getOrCreate()

machine_json_df = spark.read \
    .option("multiLine", True) \
    .json("/content/machine_configurations.json")


machine_json_df.show(5, truncate=False)

machine_json_df.printSchema()

+-----------------+--------------+----------+------------+----------------+--------+----------------------------------------------------------------------+--------------+
|capacity_per_hour|installed_year|machine_id|machine_type|maintenance     |plant_id|sensors                                                               |specifications|
+-----------------+--------------+----------+------------+----------------+--------+----------------------------------------------------------------------+--------------+
|120              |2022          |MC001     |CNC         |{60, 2026-01-14}|PL01    |[{true, temperature, C}, {true, vibration, mm/s}, {true, energy, kWh}]|{85, 22, 5.0} |
|80               |2024          |MC002     |Assembly    |{60, 2026-05-19}|PL01    |[{true, temperature, C}, {true, vibration, mm/s}, {true, energy, kWh}]|{85, 22, 5.5} |
|120              |2018          |MC003     |Casting     |{45, 2026-08-28}|PL01    |[{true, temperature, C}, {true, vibration, mm/s}, {true, ener

In [206]:
from pyspark.sql.functions import col

machine_json_df.select(
    col("machine_id"),
    col("plant_id"),
    col("machine_type"),
    col("specifications.rated_power_kw").alias("rated_power_kw"),
    col("specifications.max_temperature_c").alias("max_temperature_c"),
    col("specifications.vibration_limit_mm_s").alias("vibration_limit_mm_s")).show(10, truncate=False)

+----------+--------+------------+--------------+-----------------+--------------------+
|machine_id|plant_id|machine_type|rated_power_kw|max_temperature_c|vibration_limit_mm_s|
+----------+--------+------------+--------------+-----------------+--------------------+
|MC001     |PL01    |CNC         |22            |85               |5.0                 |
|MC002     |PL01    |Assembly    |22            |85               |5.5                 |
|MC003     |PL01    |Casting     |55            |90               |4.5                 |
|MC004     |PL01    |Welding     |22            |85               |4.5                 |
|MC005     |PL01    |Testing     |30            |85               |4.5                 |
|MC006     |PL02    |CNC         |30            |85               |5.5                 |
|MC007     |PL02    |Assembly    |22            |80               |4.5                 |
|MC008     |PL02    |Casting     |55            |75               |5.0                 |
|MC009     |PL02    |

In [210]:
machine_maintenance_df = machine_json_df.select(
    col("machine_id"),
    col("maintenance.frequency_days").alias("frequency_days"),
    col("maintenance.last_service").alias("last_service"))

machine_maintenance_df.show(10, truncate=False)

+----------+--------------+------------+
|machine_id|frequency_days|last_service|
+----------+--------------+------------+
|MC001     |60            |2026-01-14  |
|MC002     |60            |2026-05-19  |
|MC003     |45            |2026-08-28  |
|MC004     |60            |2026-04-19  |
|MC005     |30            |2026-05-09  |
|MC006     |60            |2026-01-09  |
|MC007     |30            |2026-06-26  |
|MC008     |30            |2026-04-21  |
|MC009     |30            |2026-08-13  |
|MC010     |45            |2026-06-05  |
+----------+--------------+------------+
only showing top 10 rows


In [211]:
machine_sensors_df = machine_json_df.select(
    col("machine_id"),
    col("sensors"))

machine_sensors_df.show(truncate=False)

+----------+----------------------------------------------------------------------+
|machine_id|sensors                                                               |
+----------+----------------------------------------------------------------------+
|MC001     |[{true, temperature, C}, {true, vibration, mm/s}, {true, energy, kWh}]|
|MC002     |[{true, temperature, C}, {true, vibration, mm/s}, {true, energy, kWh}]|
|MC003     |[{true, temperature, C}, {true, vibration, mm/s}, {true, energy, kWh}]|
|MC004     |[{true, temperature, C}, {true, vibration, mm/s}, {true, energy, kWh}]|
|MC005     |[{true, temperature, C}, {true, vibration, mm/s}, {true, energy, kWh}]|
|MC006     |[{true, temperature, C}, {true, vibration, mm/s}, {true, energy, kWh}]|
|MC007     |[{true, temperature, C}, {true, vibration, mm/s}, {true, energy, kWh}]|
|MC008     |[{true, temperature, C}, {true, vibration, mm/s}, {true, energy, kWh}]|
|MC009     |[{true, temperature, C}, {true, vibration, mm/s}, {true, energy,

In [213]:
from pyspark.sql.functions import explode

machine_sensor_rows_df = machine_json_df.select(
    col("machine_id"),
    explode(col("sensors")).alias("sensor"))
machine_sensor_rows_df.show(20, truncate=False)

+----------+-----------------------+
|machine_id|sensor                 |
+----------+-----------------------+
|MC001     |{true, temperature, C} |
|MC001     |{true, vibration, mm/s}|
|MC001     |{true, energy, kWh}    |
|MC002     |{true, temperature, C} |
|MC002     |{true, vibration, mm/s}|
|MC002     |{true, energy, kWh}    |
|MC003     |{true, temperature, C} |
|MC003     |{true, vibration, mm/s}|
|MC003     |{true, energy, kWh}    |
|MC004     |{true, temperature, C} |
|MC004     |{true, vibration, mm/s}|
|MC004     |{true, energy, kWh}    |
|MC005     |{true, temperature, C} |
|MC005     |{true, vibration, mm/s}|
|MC005     |{true, energy, kWh}    |
|MC006     |{true, temperature, C} |
|MC006     |{true, vibration, mm/s}|
|MC006     |{true, energy, kWh}    |
|MC007     |{true, temperature, C} |
|MC007     |{true, vibration, mm/s}|
+----------+-----------------------+
only showing top 20 rows


In [218]:
from pyspark.sql.functions import count

sensor_count_df = machine_sensor_rows_df.groupBy(
    "sensor").agg(
    count("machine_id").alias("machine_count"))

sensor_count_df.show(truncate=False)

+-----------------------+-------------+
|sensor                 |machine_count|
+-----------------------+-------------+
|{true, energy, kWh}    |30           |
|{true, temperature, C} |30           |
|{true, vibration, mm/s}|30           |
+-----------------------+-------------+



In [220]:
machine_high_limits_df = machine_json_df.filter(
    (col("specifications.max_temperature_c") > 75) |
    (col("specifications.vibration_limit_mm_s") > 5))

machine_high_limits_df.select(
    "machine_id",
    "machine_type",
    "specifications.max_temperature_c",
    "specifications.vibration_limit_mm_s").show(truncate=False)

+----------+------------+-----------------+--------------------+
|machine_id|machine_type|max_temperature_c|vibration_limit_mm_s|
+----------+------------+-----------------+--------------------+
|MC001     |CNC         |85               |5.0                 |
|MC002     |Assembly    |85               |5.5                 |
|MC003     |Casting     |90               |4.5                 |
|MC004     |Welding     |85               |4.5                 |
|MC005     |Testing     |85               |4.5                 |
|MC006     |CNC         |85               |5.5                 |
|MC007     |Assembly    |80               |4.5                 |
|MC009     |Welding     |85               |5.5                 |
|MC010     |Testing     |90               |4.5                 |
|MC011     |CNC         |90               |6.0                 |
|MC012     |Assembly    |80               |5.5                 |
|MC013     |Casting     |90               |5.0                 |
|MC014     |Welding     |

In [221]:
from pyspark.sql.functions import col

machine_flat_df = machine_json_df.select(
    col("machine_id"),
    col("plant_id"),
    col("machine_type"),
    col("installed_year"),
    col("capacity_per_hour"),
    col("specifications.rated_power_kw").alias("rated_power_kw"),
    col("specifications.max_temperature_c").alias("max_temperature_c"),
    col("specifications.vibration_limit_mm_s").alias("vibration_limit_mm_s"),
    col("maintenance.frequency_days").alias("frequency_days"),
    col("maintenance.last_service").alias("last_service"))

machine_flat_df.show(10, truncate=False)

machine_flat_df.printSchema()

+----------+--------+------------+--------------+-----------------+--------------+-----------------+--------------------+--------------+------------+
|machine_id|plant_id|machine_type|installed_year|capacity_per_hour|rated_power_kw|max_temperature_c|vibration_limit_mm_s|frequency_days|last_service|
+----------+--------+------------+--------------+-----------------+--------------+-----------------+--------------------+--------------+------------+
|MC001     |PL01    |CNC         |2022          |120              |22            |85               |5.0                 |60            |2026-01-14  |
|MC002     |PL01    |Assembly    |2024          |80               |22            |85               |5.5                 |60            |2026-05-19  |
|MC003     |PL01    |Casting     |2018          |120              |55            |90               |4.5                 |45            |2026-08-28  |
|MC004     |PL01    |Welding     |2024          |180              |22            |85               |

In [222]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("UnionAndUnionAll") \
    .getOrCreate()


df_main = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("/content/manufacturing_production_120k.csv")


df_extra = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("/content/manufacturing_production_extra_12k.csv")

print("Main Production Dataset")
df_main.show(5, truncate=False)

print("Extra Production Dataset")
df_extra.show(5, truncate=False)
print("Main dataset rows:", df_main.count())
print("Extra dataset rows:", df_extra.count())

Main Production Dataset
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|shift  |product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|PR0000001    |2026-08-07     |PL03    |MC011     |OP0275     |Evening|MTR-100     |Motors        |1060         |993           |30             |34              |388.11    |61.4         |3.88          |Rework        |No              |Charan  

In [223]:
print("Main Dataset Schema")
df_main.printSchema()

print("Extra Dataset Schema")
df_extra.printSchema()

print("Main columns:", df_main.columns)
print("Extra columns:", df_extra.columns)

if df_main.columns == df_extra.columns:
    print("Column names and order match!")
else:
    print("Column names or order differ. Check before union.")

Main Dataset Schema
root
 |-- production_id: string (nullable = true)
 |-- production_date: date (nullable = true)
 |-- plant_id: string (nullable = true)
 |-- machine_id: string (nullable = true)
 |-- operator_id: string (nullable = true)
 |-- shift: string (nullable = true)
 |-- product_code: string (nullable = true)
 |-- product_family: string (nullable = true)
 |-- units_planned: integer (nullable = true)
 |-- units_produced: integer (nullable = true)
 |-- defective_units: integer (nullable = true)
 |-- downtime_minutes: integer (nullable = true)
 |-- energy_kwh: double (nullable = true)
 |-- temperature_c: double (nullable = true)
 |-- vibration_mm_s: double (nullable = true)
 |-- quality_status: string (nullable = true)
 |-- maintenance_flag: string (nullable = true)
 |-- supervisor: string (nullable = true)

Extra Dataset Schema
root
 |-- production_id: string (nullable = true)
 |-- production_date: date (nullable = true)
 |-- plant_id: string (nullable = true)
 |-- machine_id: 

In [224]:

df_union_all = df_main.union(df_extra)


df_union_all.show(10, truncate=False)


print("Main dataset rows:", df_main.count())
print("Extra dataset rows:", df_extra.count())
print("Combined dataset rows:", df_union_all.count())

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|shift  |product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|PR0000001    |2026-08-07     |PL03    |MC011     |OP0275     |Evening|MTR-100     |Motors        |1060         |993           |30             |34              |388.11    |61.4         |3.88          |Rework        |No              |Charan    |
|PR0000002    |2026-

In [225]:

data1 = [
    ("PR001", "PL01", 100),
    ("PR002", "PL02", 200)]

columns1 = [
    "production_id",
    "plant_id",
    "units_produced"]

df1 = spark.createDataFrame(data1, columns1)

data2 = [
    ("PL03", "PR003", 300),
    ("PL04", "PR004", 400)]

columns2 = [
    "plant_id",
    "production_id",
    "units_produced"]

df2 = spark.createDataFrame(data2, columns2)

print("First DataFrame")
df1.show()

print("Second DataFrame")
df2.show()

df_combined = df1.unionByName(df2)

print("Combined DataFrame")
df_combined.show()

First DataFrame
+-------------+--------+--------------+
|production_id|plant_id|units_produced|
+-------------+--------+--------------+
|        PR001|    PL01|           100|
|        PR002|    PL02|           200|
+-------------+--------+--------------+

Second DataFrame
+--------+-------------+--------------+
|plant_id|production_id|units_produced|
+--------+-------------+--------------+
|    PL03|        PR003|           300|
|    PL04|        PR004|           400|
+--------+-------------+--------------+

Combined DataFrame
+-------------+--------+--------------+
|production_id|plant_id|units_produced|
+-------------+--------+--------------+
|        PR001|    PL01|           100|
|        PR002|    PL02|           200|
|        PR003|    PL03|           300|
|        PR004|    PL04|           400|
+-------------+--------+--------------+

